In [ ]:
import os
import sys
import json
import csv
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
import geopandas as gpd
import h3
import h3pandas

from shapely import wkt
from shapely.geometry import shape, Polygon
from shapely.geometry.polygon import orient
from shapely.ops import unary_union
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
# Get the parent directory of the current directory (which is 'notebooks')
# and add it to the system path
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    
if project_root not in sys.path:
    sys.path.append(project_root)

# Now you can use absolute imports from the project root
from utils.geometry import get_bearing, get_bearing_label, convert_geometry

In [ ]:
d_drive_root = "D:\\"
if d_drive_root not in sys.path:
    sys.path.append(d_drive_root)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def get_border_type_from_name(NAME, NAMELSAD):
    if NAME.endswith('(balance)'):
        return 'balance'
        
    elif NAME.endswith('County'):
        return 'county'
    
    last_word = NAMELSAD[len(NAME):].strip()
    # print(last_word)

    if last_word == '' or last_word == 'CDP':
        return 'census_designated_place'

    # ['(state)_reservation',
    #  'reservation',
    #  'indian_reservation',
    #  'rancheria',
    #  'census_designated_place',
    #  'colony',
    #  'pueblo',
    #  'indian_rancheria',
    #  'f_pojoaque',
    #  'indian_colony',
    #  'community',
    #  'indian_community',
    #  'reserve',
    #  'ranch',
    #  'village',
    #  'de_cochiti',
    #  'indian_village',
    #  'settlement',
    #  'trust_land',
    #  'off-reservation_trust_land',
    #  'hawaiian_home_land',
    #  'anvsa',
    #  'otsa',
    #  'sdtsa',
    #  'tdsa',
    #  'joint-use_otsa',
    #  'joint-use_area']

    if last_word.startswith('(state)'):
        last_word = last_word.replace('(', '')
        last_word = last_word.replace(')', '')

    if '-' in last_word:
        last_word = last_word.replace('-', '_')

    # sanitize
    clean_last_word = last_word.strip().lower()
    clean_last_word = clean_last_word.replace(' ', '_')   
        
    return clean_last_word

### Geo Admin boundaries

In [ ]:
# YEAR = 2024

In [ ]:
YEAR = 2020

In [ ]:
DATA_DIR = os.path.join(d_drive_root, os.path.join("census", str(YEAR)))

In [ ]:
DATA_DIR

#### Get Census Tracts

In [ ]:
census_tract_gdf = gpd.GeoDataFrame()
filter_fn_type = 'tract'
for fn in os.listdir(DATA_DIR):
    use_type = f"_{filter_fn_type}.zip"
    if fn.endswith(use_type):
        full_path = os.path.join(DATA_DIR, fn)
        tmp_fn_gdf = gpd.GeoDataFrame.from_file(full_path)
        census_tract_gdf = pd.concat([census_tract_gdf, tmp_fn_gdf], axis=0)
        

In [ ]:
census_tract_gdf = census_tract_gdf.filter(["GEOID", "NAMELSAD", "geometry"])

In [ ]:
census_tract_gdf = census_tract_gdf.rename(columns={"GEOID": "geoid", "NAMELSAD": "name"})

In [ ]:
census_tract_gdf["border_type"] = "tract"
census_tract_gdf["border_subtype"] = None

In [ ]:
census_tract_gdf

#### Get Census places

In [ ]:
census_place_gdf = gpd.GeoDataFrame()
filter_fn_type = 'place'
for fn in os.listdir(DATA_DIR):
    use_type = f"_{filter_fn_type}.zip"
    if fn.endswith(use_type):
        tmp_fn_gdf = gpd.GeoDataFrame.from_file(os.path.join(DATA_DIR, fn))
        census_place_gdf = pd.concat([census_place_gdf, tmp_fn_gdf], axis=0)
        

In [ ]:
census_place_gdf = census_place_gdf.filter(["GEOID", "NAME", "NAMELSAD", "geometry"])

In [ ]:
census_place_gdf["border_type"] = "place"
census_place_gdf["border_subtype"] = census_place_gdf[["NAME", "NAMELSAD"]].apply(lambda x: get_border_type_from_name(**x), axis=1)

#### Get Census Counties

In [ ]:
county_filename = f"tl_{str(YEAR)}_us_county.zip"
county_fn = os.path.join(DATA_DIR, county_filename)
county_gdf = gpd.GeoDataFrame.from_file(county_fn)

In [ ]:
county_gdf = county_gdf.filter(["GEOID", "NAME", "NAMELSAD", "geometry"])
county_gdf["border_type"] = "county"
county_gdf['border_subtype'] = county_gdf[["NAME", "NAMELSAD"]].apply(lambda x: get_border_type_from_name(**x), axis=1)

#### Combine Census Places + Counties

In [ ]:
# set name for county set only using namelsad
county_gdf = county_gdf.rename(columns={"NAMELSAD": "name", "GEOID": "geoid"})
del county_gdf["NAME"]

census_place_gdf = census_place_gdf.rename(columns={"GEOID": "geoid", "NAME": "name"})
del census_place_gdf["NAMELSAD"]

census_gdf = pd.concat([census_place_gdf, county_gdf], axis=0)

#### Include Census Tracts

In [ ]:
census_gdf = pd.concat([census_gdf, census_tract_gdf], axis=0)

#### Get Military Zones

In [ ]:
mil_gdf = gpd.GeoDataFrame.from_file(os.path.join(DATA_DIR, f"tl_{str(YEAR)}_us_mil.zip"))

In [ ]:
mil_gdf = mil_gdf.filter(["AREAID", "FULLNAME", "geometry"])

In [ ]:
mil_gdf = mil_gdf.rename(columns={"AREAID": "geoid", "FULLNAME": "name"})

In [ ]:
mil_gdf["border_type"] = "military_zone"

In [ ]:
mil_gdf["border_subtype"] = None

In [ ]:
census_gdf = pd.concat([census_gdf, mil_gdf], axis=0)

#### Get Native lands

In [ ]:
aiannh_gdf = gpd.GeoDataFrame.from_file(os.path.join(DATA_DIR, f"tl_{str(YEAR)}_us_aiannh.zip"))

In [ ]:
aiannh_gdf = aiannh_gdf.filter(["GEOID", "NAME", "NAMELSAD", "geometry"])

In [ ]:
aiannh_gdf["border_type"] = "native_land"

In [ ]:
aiannh_gdf["border_subtype"] = aiannh_gdf[["NAME", "NAMELSAD"]].apply(lambda x: get_border_type_from_name(**x), axis=1)

In [ ]:
aiannh_gdf = aiannh_gdf.rename(columns={"GEOID": "geoid", "NAME": "name"})

In [ ]:
del aiannh_gdf["NAMELSAD"]

In [ ]:
census_gdf = pd.concat([census_gdf, aiannh_gdf], axis=0)

In [ ]:
census_gdf.border_subtype.drop_duplicates().to_list()

In [ ]:
max_len = 0
name_max_len = ''
for n in census_gdf.name.drop_duplicates().to_list():
    if len(n) > max_len:
        name_max_len = n
        max_len = len(n)

In [ ]:
max_len = 0
geoid_max_len = ''
for n in census_gdf.geoid.drop_duplicates().to_list():
    if len(n) > max_len:
        geoid_max_len = n
        max_len = len(n)

In [ ]:
max_len

In [ ]:
max_len = 0
border_subtype_max_len = ''
for n in census_gdf.border_subtype.drop_duplicates().to_list():
    if n is None:
        continue
    if len(n) > max_len:
        border_subtype_max_len = n
        max_len = len(n)

In [ ]:
max_len

In [ ]:
census_gdf["year"] = YEAR

In [ ]:
# census_gdf = gpd.GeoDataFrame.from_file(data_fn)

In [ ]:
census_gdf.plot(figsize=(13,8))

In [ ]:
# --- Save the GeoDataFrame to a File ---
output_file = f"public_borders_{str(YEAR)}.geojson"

try:
    census_gdf.to_file(output_file, driver='GeoJSON')
    print("-" * 50)
    print(f"Successfully created GeoJSON file: {output_file}")
    print(f"File saved with {len(census_gdf)} features.")
    print("You can now load this file into QGIS to visualize your H3 coverage areas.")
except Exception as e:
    print(f"Error saving file: {e}")

# Display a preview of the GeoDataFrame structure
print("\nGeoDataFrame Preview:")
print(census_gdf.head())

#### Create a 10km buffer for safety

In [ ]:
census_gdf = census_gdf.to_crs(3857)

In [ ]:
census_gdf['geom_buffered'] = census_gdf.buffer(10000)  # buffer extends bounds by 10km

In [ ]:
census_gdf["border_area_m"] = census_gdf.to_crs(crs="EPSG:6933").geometry.area

In [ ]:
census_gdf["border_area"] = census_gdf["border_area_m"] / 1e6

In [ ]:
census_gdf

In [ ]:
census_gdf = census_gdf.to_crs(4326)

### H3

In [ ]:
gdf_for_polyfill = census_gdf.set_geometry('geom_buffered')

In [ ]:
empty_geoms = gdf_for_polyfill.geometry.is_empty
print(f"Found {empty_geoms.sum()} empty geometries.")

In [ ]:
invalid_geoms = ~gdf_for_polyfill.geometry.is_valid
print(f"Found {invalid_geoms.sum()} invalid geometries.")

In [ ]:
is_valid_and_not_empty = gdf_for_polyfill.geometry.is_valid & ~gdf_for_polyfill.geometry.is_empty
cleaned_gdf = gdf_for_polyfill[is_valid_and_not_empty]

In [ ]:
print(f"Original number of rows: {len(gdf_for_polyfill)}")
print(f"Number of rows after cleaning: {len(cleaned_gdf)}")

In [ ]:
if not cleaned_gdf.empty:
    hex_ids_df = cleaned_gdf.h3.polyfill(5, explode=True)
    # This result should be free of NaNs
else:
    print("GeoDataFrame is empty after cleaning, cannot run polyfill.")

In [ ]:
# hex_ids = gdf_for_polyfill.h3.polyfill(5, explode=True)

In [ ]:
hex_ids_df

In [ ]:
hex_ids = hex_ids_df.h3_polyfill.drop_duplicates().to_list()

In [ ]:
h5_cleaned_list = [x for x in list(set(hex_ids)) if str(x) != 'nan']

In [ ]:
hex_df = pd.DataFrame([{"hex_id": x, "border_type": "CDP"} for x in h5_cleaned_list])

In [ ]:
hex_census_df = hex_df.set_index('hex_id')

In [ ]:
hex_census_df = hex_census_df.h3.h3_to_geo_boundary()

In [ ]:
hex_census_df = hex_census_df.h3.cell_area(unit='km^2')

In [ ]:
hex_census_df.reset_index(inplace=True)

In [ ]:
hex_census_overlap_gdf = gpd.overlay(hex_census_df,\
                               census_gdf[["name", "geoid", "geometry", "border_area"]],\
                               how="intersection",\
                               keep_geom_type=False\
                              )

In [ ]:
hex_census_overlap_gdf["overlap_area_m"] = hex_census_overlap_gdf.to_crs(crs="EPSG:6933").geometry.area

In [ ]:
hex_census_overlap_gdf["overlap_area"] = hex_census_overlap_gdf["overlap_area_m"] / 1e6

In [ ]:
hex_census_overlap_gdf["overlap_ratio_hex_to_border"] = hex_census_overlap_gdf[["overlap_area", "h3_cell_area"]].apply(lambda x: min(1, x[0]/x[1]), axis=1)

In [ ]:
hex_census_overlap_gdf["overlap_ratio_border_to_hex"] = hex_census_overlap_gdf[["overlap_area", "border_area"]].apply(lambda x: min(1, x[0]/x[1]), axis=1)

In [ ]:
hex_census_overlap_gdf.overlap_ratio_hex_to_border.max()

In [ ]:
hex_census_overlap_gdf.overlap_ratio_border_to_hex.max()

In [ ]:
try:
    hex_census_overlap_gdf = hex_census_overlap_gdf.rename(columns={"NAME": "name", "GEOID": "geoid"})
except Exception as e:
    print(e)

In [ ]:
save_overlap_gdf = hex_census_overlap_gdf.filter(['hex_id', 'border_type', 'name', 'geoid', 'geometry', 'overlap_ratio_hex_to_border', 'overlap_ratio_border_to_hex'])

#### Output

In [ ]:
save_overlap_gdf['year'] = YEAR

In [ ]:
# --- 6. Save the GeoDataFrame to a File ---
output_file = f"hex_census_border_overlap_{str(YEAR)}.geojson"

# GeoJSON is generally the most compatible and robust format for sharing.
# For a shapefile, you would use: gdf.to_file('h3_coverage_validation.shp')
try:
    save_overlap_gdf.to_file(output_file, driver='GeoJSON')
    print("-" * 50)
    print(f"Successfully created GeoJSON file: {output_file}")
    print(f"File saved with {len(save_overlap_gdf)} features.")
    print("You can now load this file into QGIS to visualize your H3 coverage areas.")
except Exception as e:
    print(f"Error saving file: {e}")

# Display a preview of the GeoDataFrame structure
print("\nGeoDataFrame Preview:")
print(save_overlap_gdf.head())